<a href="https://colab.research.google.com/github/TBGhorbanpour/Social-Awareness/blob/main/PoliticalTweetFilter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
data = pd.read_csv('/content/drive/MyDrive/DBF0.8.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data.shape

(93179, 17)

**extract word_frequency file**

In [ ]:
import pandas as pd
from collections import Counter

# Assuming 'tokenized_words' is the column containing tokenized words
tokenized_words_column = 'tokenized_tweet'

# Combine all tokenized words into a single list
all_tokenized_words = [word for words in data[tokenized_words_column].dropna() for word in words.split()]

# Count the frequency of each word
word_frequencies = Counter(all_tokenized_words)

# Convert the frequencies into a DataFrame
word_frequencies_df = pd.DataFrame(list(word_frequencies.items()), columns=['Word', 'Frequency'])

# Sort the DataFrame by frequency in descending order
word_frequencies_df = word_frequencies_df.sort_values(by='Frequency', ascending=False)

# Save the word frequencies to a CSV file
word_frequencies_df.to_csv('word_frequencies.csv', sep=',', index=False, encoding='utf-8-sig')

**Filter, delelte Political tweets**

In [ ]:
import pandas as pd


# Load words from the text file
with open('political_words.txt', 'r') as file:
    words_to_exclude = set(line.strip() for line in file)

# Assuming 'tokenized_words' is the column containing tokenized words
tokenized_words_column = 'tokenized_tweet'

# Filter out rows containing words from the text file
filtered_data = data[~data[tokenized_words_column].str.split().apply(set).apply(lambda x: bool(x & words_to_exclude))]

# Save the filtered data to a new CSV file
filtered_data.to_csv('filtered_dataset.csv', index=False)

store Deleted political tweets in a dataframe

In [ ]:
# Create a DataFrame for the deleted rows
deleted_data = data[data[tokenized_words_column].str.split().apply(set).apply(lambda x: bool(x & words_to_exclude))]

In [ ]:
filtered_data.shape

(75473, 17)

In [ ]:
deleted_data.shape

(17706, 17)

**the deleted political tweets have Ecological words**

In [ ]:
# Load words from the second text file (words for counting)
with open('Ecological_words.txt', 'r') as file:
    words_to_count = set(line.strip() for line in file)

# Create a DataFrame for the deleted rows containing words from the count text file
deleted_with_count_words = deleted_data[deleted_data[tokenized_words_column].str.split().apply(set).apply(lambda x: bool(x & words_to_count))]

In [ ]:
deleted_with_count_words.shape

(17693, 17)

**Delete the Political words from dataset**

In [ ]:
# Save the indices of the rows to be deleted
rows_to_delete = deleted_data.index

# Remove rows from the original data DataFrame
data = data.drop(rows_to_delete)

In [ ]:
data.shape

(75473, 17)

**Save on google drive**

In [ ]:
from googleapiclient.http import MediaFileUpload
from googleapiclient.discovery import build

filtered_data.to_csv('filtered_Political_words.csv', sep=',', index=False, encoding='utf-8-sig')
# Saving the file to Google drive
file_name = "filtered_Political_words.csv"

from google.colab import auth
auth.authenticate_user()
drive_service = build('drive', 'v3')

def save_file_to_drive(name, path):
  file_metadata = {'name': name, 'mimeType': 'application/octet-stream'}
  media = MediaFileUpload(path, mimetype='application/octet-stream', resumable=True)
  created = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()

  return created

save_file_to_drive(file_name, file_name)


{'id': '1xKiwUL6H-FEnwl5WAJblUKbiX1cyGpVY'}